In [5]:
import os
import pandas as pd
import random

# Set random seed for reproducibility
random.seed(42)

# Paths
sessions_dir = "/Users/merveastekin/Desktop/iExplain-logAnalysis/data/HDFS_385_balanced_sampled_sessions"
ground_truth_path = "/Users/merveastekin/Desktop/iExplain-logAnalysis/data/HDFS_anomaly_label_385_sampled_balanced.csv"
output_sessions_dir = "/Users/merveastekin/Desktop/iExplain-logAnalysis/data/HDFS_10_balanced_sampled_sessions"
output_ground_truth_path = "/Users/merveastekin/Desktop/iExplain-logAnalysis/data/HDFS_anomaly_label_10_sampled_balanced.csv"

# Create output directory
os.makedirs(output_sessions_dir, exist_ok=True)

# Load ground truth data
ground_truth_df = pd.read_csv(ground_truth_path)

# Sample 100 sessions
sampled_df = ground_truth_df.sample(n=10, random_state=42)

# Save subsampled ground truth
sampled_df.to_csv(output_ground_truth_path, index=False)

# Get list of sampled session IDs (assuming first column is session ID)
sampled_session_ids = sampled_df.iloc[:, 0].tolist()

# Copy corresponding session files
for session_id in sampled_session_ids:
    src_file = os.path.join(sessions_dir, f"{session_id}.log")  # Add .log extension
    dst_file = os.path.join(output_sessions_dir, f"{session_id}.log")
    if os.path.exists(src_file):
        with open(src_file, 'r') as f:
            content = f.read()
        with open(dst_file, 'w') as f:
            f.write(content)
    else:
        print(f"Warning: File not found: {src_file}")  # helpful for debugging

print(f"Subsampled 10 sessions to {output_sessions_dir}")
print(f"Subsampled ground truth saved to {output_ground_truth_path}")

Subsampled 10 sessions to /Users/merveastekin/Desktop/iExplain-logAnalysis/data/HDFS_10_balanced_sampled_sessions
Subsampled ground truth saved to /Users/merveastekin/Desktop/iExplain-logAnalysis/data/HDFS_anomaly_label_10_sampled_balanced.csv


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

sns.set(style="whitegrid")

# -----------------------------
# Experimental Results
# -----------------------------
data = [
    ["SIMPLIFIED", "Few", "Two-Agent", 57, 51.72, 97.83, 67.67, 45, 42, 12, 1],
    ["SIMPLIFIED", "Zero", "Two-Agent", 53, 49.33, 80.43, 61.16, 37, 38, 16, 9],

    ["DETAILED", "Few", "Two-Agent", 57, 51.90, 89.13, 65.60, 41, 38, 16, 5],
    ["DETAILED", "Zero", "Two-Agent", 54, 50.00, 78.26, 61.02, 36, 36, 18, 10],

    ["MINIMAL_GENERAL", "Few", "Two-Agent", 49, 46.75, 78.26, 58.54, 36, 41, 13, 10],
    ["MINIMAL_GENERAL", "Zero", "Two-Agent", 62, 56.67, 73.91, 64.15, 34, 26, 28, 12],

    ["MINIMAL_GENERAL (Parser Few)", "Zero", "Two-Agent", 60, 55.17, 69.57, 61.54, 32, 26, 28, 14],

    ["HDFS", "Few", "Two-Agent", 56, 51.11, 100, 67.65, 46, 44, 10, 0],
    ["HDFS", "Zero", "Two-Agent", 57, 51.69, 100, 68.15, 46, 43, 11, 0],

    ["CoT", "Few", "Two-Agent", 57, 51.69, 100, 68.15, 46, 43, 11, 0],
    ["CoT", "Zero", "Two-Agent", 52, 48.84, 91.30, 63.64, 42, 44, 10, 4],

    ["OLD", "Few", "Two-Agent", 56, 51.52, 73.91, 60.71, 34, 32, 22, 12],
    ["OLD", "Zero", "Two-Agent", 65, 59.65, 73.91, 66.02, 34, 23, 31, 12],

    ["SingleAgent", "Few", "Single-Agent", 65, 61.22, 65.22, 63.16, 30, 19, 35, 16],
    ["SingleAgent", "Zero", "Single-Agent", 75, 83.87, 56.52, 67.53, 26, 5, 49, 20],

    ["NoAgent", "Few", "No-Agent", 73, 69.39, 73.91, 71.58, 34, 15, 39, 12],
    ["NoAgent", "Zero", "No-Agent", 74, 81.25, 56.52, 66.67, 26, 6, 48, 20],
]

columns = [
    "Prompt", "Shot", "Architecture",
    "Accuracy", "Precision", "Recall", "F1",
    "TP", "FP", "TN", "FN"
]

df = pd.DataFrame(data, columns=columns)

# Label for plotting
df["Experiment"] = df["Prompt"] + " (" + df["Shot"] + ")"

# -----------------------------
# Create PDF with plots
# -----------------------------
with PdfPages("log_analysis_results.pdf") as pdf:

    # 1️⃣ Main Metrics Comparison
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    metrics = ["Accuracy", "Precision", "Recall", "F1"]

    for ax, metric in zip(axes.flatten(), metrics):
        sns.barplot(
            data=df,
            x="Experiment",
            y=metric,
            hue="Architecture",
            ax=ax
        )
        ax.set_title(metric)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right")

    plt.tight_layout()
    pdf.savefig()
    plt.close()


    # 2️⃣ Few vs Zero Comparison
    fig, axes = plt.subplots(1, 2, figsize=(12,5))

    sns.boxplot(data=df, x="Shot", y="F1", ax=axes[0])
    axes[0].set_title("F1 Score Distribution: Few vs Zero Shot")

    sns.boxplot(data=df, x="Shot", y="Accuracy", ax=axes[1])
    axes[1].set_title("Accuracy Distribution: Few vs Zero Shot")

    plt.tight_layout()
    pdf.savefig()
    plt.close()


    # 3️⃣ Architecture Comparison
    fig, axes = plt.subplots(1, 2, figsize=(12,5))

    sns.barplot(data=df, x="Architecture", y="F1", ax=axes[0])
    axes[0].set_title("F1 Score by Architecture")

    sns.barplot(data=df, x="Architecture", y="Accuracy", ax=axes[1])
    axes[1].set_title("Accuracy by Architecture")

    plt.tight_layout()
    pdf.savefig()
    plt.close()


    # 4️⃣ Confusion Matrix Components
    fig, axes = plt.subplots(2, 2, figsize=(12,10))

    components = ["TP", "FP", "TN", "FN"]

    for ax, comp in zip(axes.flatten(), components):
        sns.barplot(data=df, x="Experiment", y=comp, ax=ax)
        ax.set_title(comp)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right")

    plt.tight_layout()
    pdf.savefig()
    plt.close()


print("PDF saved as log_analysis_results.pdf")

/var/folders/c4/sb673w6n7g78z446d1_0hrzw0000gn/T/ipykernel_24307/4194810939.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right")
/var/folders/c4/sb673w6n7g78z446d1_0hrzw0000gn/T/ipykernel_24307/4194810939.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right")
/var/folders/c4/sb673w6n7g78z446d1_0hrzw0000gn/T/ipykernel_24307/4194810939.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right")
/var/folders/c4/sb673w6n7g78z446d1_0hrzw0000gn/T/ipykernel_24307/4194810939.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, 

PDF saved as log_analysis_results.pdf


/var/folders/c4/sb673w6n7g78z446d1_0hrzw0000gn/T/ipykernel_24307/4194810939.py:112: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right")
/var/folders/c4/sb673w6n7g78z446d1_0hrzw0000gn/T/ipykernel_24307/4194810939.py:112: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right")
/var/folders/c4/sb673w6n7g78z446d1_0hrzw0000gn/T/ipykernel_24307/4194810939.py:112: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right")
/var/folders/c4/sb673w6n7g78z446d1_0hrzw0000gn/T/ipykernel_24307/4194810939.py:112: UserWarning: set_ticklabels() should only be used with a fixed number of tic

In [8]:
"""
iExplain — Log Analysis Experiment Results
Generates 6 chart PDFs matching the HTML dashboard.
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import os

# ── Output directory ──────────────────────────────────────────────────────────
OUT_DIR = "/Users/merveastekin/Desktop/iExplain-logAnalysis/plots"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Colour palette (matches dashboard) ───────────────────────────────────────
C_ACCENT  = "#5effd8"   # teal
C_PREC    = "#818cf8"   # purple
C_REC     = "#ff5e8a"   # pink/red
C_F1      = "#ffe05e"   # amber
C_SA      = "#fbbf24"   # yellow (Single Agent)
C_NA      = "#34d399"   # green  (No Agent)
C_FEW     = C_ACCENT
C_ZERO    = C_REC
BG        = "#0a0c10"
SURFACE   = "#111318"
SURFACE2  = "#181b22"
GRID      = "#252830"
TEXT      = "#e8ecf0"
TEXT_DIM  = "#6b7280"
TEXT_MID  = "#9ca3af"

def apply_dark_style(fig, axes=None):
    fig.patch.set_facecolor(BG)
    if axes is None:
        axes = fig.get_axes()
    for ax in (axes if hasattr(axes, '__iter__') else [axes]):
        ax.set_facecolor(SURFACE)
        ax.tick_params(colors=TEXT_MID, labelsize=9)
        ax.xaxis.label.set_color(TEXT_DIM)
        ax.yaxis.label.set_color(TEXT_DIM)
        ax.title.set_color(TEXT)
        for spine in ax.spines.values():
            spine.set_edgecolor(GRID)
        ax.grid(color=GRID, linewidth=0.6, linestyle="--", alpha=0.8)
        ax.set_axisbelow(True)

# ── Data ──────────────────────────────────────────────────────────────────────
data = [
    # prompt,           agent, shot,    acc,   prec,   rec,    f1,    tp, fp, tn, fn
    ("SIMPLIFIED",      "DA",  "Few",   57,  51.72,  97.83, 67.67,  45, 42, 12,  1),
    ("SIMPLIFIED",      "DA",  "Zero",  53,  49.33,  80.43, 61.16,  37, 38, 16,  9),
    ("DETAILED",        "DA",  "Few",   57,  51.90,  89.13, 65.60,  41, 38, 16,  5),
    ("DETAILED",        "DA",  "Zero",  54,  50.00,  78.26, 61.02,  36, 36, 18, 10),
    ("MINIMAL_GEN",     "DA",  "Few",   49,  46.75,  78.26, 58.54,  36, 41, 13, 10),
    ("MINIMAL_GEN",     "DA",  "Zero",  62,  56.67,  73.91, 64.15,  34, 26, 28, 12),
    ("MINIMAL_GEN†",    "DA",  "Zero†", 60,  55.17,  69.57, 61.54,  32, 26, 28, 14),
    ("HDFS",            "DA",  "Few",   56,  51.11, 100.00, 67.65,  46, 44, 10,  0),
    ("HDFS",            "DA",  "Zero",  57,  51.69, 100.00, 68.15,  46, 43, 11,  0),
    ("HDFS†",           "DA",  "Zero†", 57,  51.69, 100.00, 68.15,  46, 43, 11,  0),
    ("CoT",             "DA",  "Few",   57,  51.69, 100.00, 68.15,  46, 43, 11,  0),
    ("CoT",             "DA",  "Zero",  52,  48.84,  91.30, 63.64,  42, 44, 10,  4),
    ("OLD",             "DA",  "Few",   56,  51.52,  73.91, 60.71,  34, 32, 22, 12),
    ("OLD",             "DA",  "Zero",  65,  59.65,  73.91, 66.02,  34, 23, 31, 12),
    ("OLD†",            "DA",  "Zero†", 62,  57.14,  69.57, 62.75,  32, 24, 30, 14),
    # Single Agent
    ("—",               "SA",  "Few",   65,  61.22,  65.22, 63.16,  30, 19, 35, 16),
    ("—",               "SA",  "Zero",  75,  83.87,  56.52, 67.53,  26,  5, 49, 20),
    # No Agent
    ("—",               "NA",  "Few",   73,  69.39,  73.91, 71.58,  34, 15, 39, 12),
    ("—",               "NA",  "Zero",  74,  81.25,  56.52, 66.67,  26,  6, 48, 20),
]

# Unpack into arrays
prompts = [d[0] for d in data]
agents  = [d[1] for d in data]
shots   = [d[2] for d in data]
acc     = np.array([d[3]  for d in data], dtype=float)
prec    = np.array([d[4]  for d in data], dtype=float)
rec     = np.array([d[5]  for d in data], dtype=float)
f1      = np.array([d[6]  for d in data], dtype=float)
tp      = np.array([d[7]  for d in data], dtype=float)
fp      = np.array([d[8]  for d in data], dtype=float)
tn      = np.array([d[9]  for d in data], dtype=float)
fn      = np.array([d[10] for d in data], dtype=float)

x_labels = [f"{p}\n{a}·{s}" for p, a, s in zip(prompts, agents, shots)]


# ═══════════════════════════════════════════════════════════════════════════════
# Chart 1 — All metrics bar chart
# ═══════════════════════════════════════════════════════════════════════════════
def chart1_all_metrics():
    fig, ax = plt.subplots(figsize=(18, 6))
    apply_dark_style(fig, ax)

    n   = len(data)
    x   = np.arange(n)
    w   = 0.2

    ax.bar(x - 1.5*w, acc,  width=w, color=C_ACCENT, alpha=0.85, label="Accuracy",  zorder=3)
    ax.bar(x - 0.5*w, prec, width=w, color=C_PREC,   alpha=0.85, label="Precision", zorder=3)
    ax.bar(x + 0.5*w, rec,  width=w, color=C_REC,    alpha=0.85, label="Recall",    zorder=3)
    ax.bar(x + 1.5*w, f1,   width=w, color=C_F1,     alpha=0.85, label="F1",        zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, fontsize=7, rotation=45, ha="right", color=TEXT_MID)
    ax.set_ylim(40, 108)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.set_title("Accuracy · Precision · Recall · F1 — All Configurations",
                 color=TEXT, fontsize=13, pad=14, fontweight="bold")
    ax.set_ylabel("Score (%)", color=TEXT_DIM)

    legend = ax.legend(frameon=True, fontsize=9, labelcolor=TEXT_MID)
    legend.get_frame().set_facecolor(SURFACE2)
    legend.get_frame().set_edgecolor(GRID)

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# Chart 2 — Precision vs Recall scatter
# ═══════════════════════════════════════════════════════════════════════════════
def chart2_scatter():
    fig, ax = plt.subplots(figsize=(8, 6))
    apply_dark_style(fig, ax)

    agent_cfg = {
        "DA": (C_PREC,  "o", "Dual Agent"),
        "SA": (C_SA,    "s", "Single Agent"),
        "NA": (C_NA,    "^", "No Agent"),
    }

    for ag, (color, marker, label) in agent_cfg.items():
        idx = [i for i, a in enumerate(agents) if a == ag]
        ax.scatter(prec[idx], rec[idx],
                   c=color, marker=marker, s=90, alpha=0.9,
                   label=label, zorder=4, edgecolors="white", linewidths=0.4)

    # annotate a few notable points
    notable = [
        (15, "SA·Zero"),   # best accuracy
        (17, "NA·Few"),    # best F1
        (0,  "SIMPL·Few"), # high recall
    ]
    for row_idx, lbl in notable:
        ax.annotate(lbl,
                    xy=(prec[row_idx], rec[row_idx]),
                    xytext=(6, 4), textcoords="offset points",
                    fontsize=7.5, color=TEXT_MID)

    ax.set_xlabel("Precision (%)", fontsize=10)
    ax.set_ylabel("Recall (%)",    fontsize=10)
    ax.set_title("Precision vs Recall — by Agent Type",
                 color=TEXT, fontsize=13, pad=14, fontweight="bold")
    ax.set_xlim(43, 88)
    ax.set_ylim(50, 107)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))

    legend = ax.legend(frameon=True, fontsize=9, labelcolor=TEXT_MID)
    legend.get_frame().set_facecolor(SURFACE2)
    legend.get_frame().set_edgecolor(GRID)

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# Chart 3 — Few-shot vs Zero-shot average metrics
# ═══════════════════════════════════════════════════════════════════════════════
def chart3_shot_compare():
    fig, ax = plt.subplots(figsize=(7, 5))
    apply_dark_style(fig, ax)

    few_idx  = [i for i, s in enumerate(shots) if s == "Few"]
    zero_idx = [i for i, s in enumerate(shots) if s == "Zero"]

    metrics = ["Accuracy", "Precision", "Recall", "F1"]
    few_avgs  = [np.mean(arr[few_idx])  for arr in (acc, prec, rec, f1)]
    zero_avgs = [np.mean(arr[zero_idx]) for arr in (acc, prec, rec, f1)]

    x = np.arange(len(metrics))
    w = 0.35

    ax.bar(x - w/2, few_avgs,  width=w, color=C_FEW,  alpha=0.85, label="Few-shot",  zorder=3)
    ax.bar(x + w/2, zero_avgs, width=w, color=C_ZERO, alpha=0.85, label="Zero-shot", zorder=3)

    # value labels
    for xi, (fv, zv) in enumerate(zip(few_avgs, zero_avgs)):
        ax.text(xi - w/2, fv + 0.5, f"{fv:.1f}", ha="center", va="bottom",
                fontsize=8, color=TEXT_MID)
        ax.text(xi + w/2, zv + 0.5, f"{zv:.1f}", ha="center", va="bottom",
                fontsize=8, color=TEXT_MID)

    ax.set_xticks(x)
    ax.set_xticklabels(metrics, color=TEXT_MID)
    ax.set_ylim(50, 95)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.set_title("Few-shot vs Zero-shot — Average Metrics",
                 color=TEXT, fontsize=13, pad=14, fontweight="bold")
    ax.set_ylabel("Score (%)", color=TEXT_DIM)

    legend = ax.legend(frameon=True, fontsize=9, labelcolor=TEXT_MID)
    legend.get_frame().set_facecolor(SURFACE2)
    legend.get_frame().set_edgecolor(GRID)

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# Chart 4 — Agent type accuracy comparison
# ═══════════════════════════════════════════════════════════════════════════════
def chart4_agent_accuracy():
    fig, ax = plt.subplots(figsize=(7, 5))
    apply_dark_style(fig, ax)

    agent_types = ["DA", "SA", "NA"]
    shot_types  = ["Few", "Zero"]
    x = np.arange(len(agent_types))
    w = 0.35

    for si, (shot, color) in enumerate(zip(shot_types, [C_FEW, C_ZERO])):
        avgs = []
        for ag in agent_types:
            idx = [i for i, (a, s) in enumerate(zip(agents, shots)) if a == ag and s == shot]
            avgs.append(np.mean(acc[idx]) if idx else 0)
        offset = (si - 0.5) * w
        bars = ax.bar(x + offset, avgs, width=w, color=color, alpha=0.85,
                      label=f"{shot}-shot", zorder=3)
        for bar, v in zip(bars, avgs):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.4,
                    f"{v:.1f}", ha="center", va="bottom", fontsize=8.5, color=TEXT_MID)

    ax.set_xticks(x)
    ax.set_xticklabels(["Dual Agent", "Single Agent", "No Agent"], color=TEXT_MID)
    ax.set_ylim(45, 85)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.set_title("Accuracy by Agent Architecture", color=TEXT, fontsize=13, pad=14, fontweight="bold")
    ax.set_ylabel("Accuracy (%)", color=TEXT_DIM)

    legend = ax.legend(frameon=True, fontsize=9, labelcolor=TEXT_MID)
    legend.get_frame().set_facecolor(SURFACE2)
    legend.get_frame().set_edgecolor(GRID)

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# Chart 5 — Agent type F1 comparison
# ═══════════════════════════════════════════════════════════════════════════════
def chart5_agent_f1():
    fig, ax = plt.subplots(figsize=(7, 5))
    apply_dark_style(fig, ax)

    agent_types = ["DA", "SA", "NA"]
    shot_types  = ["Few", "Zero"]
    x = np.arange(len(agent_types))
    w = 0.35

    for si, (shot, color) in enumerate(zip(shot_types, [C_FEW, C_ZERO])):
        avgs = []
        for ag in agent_types:
            idx = [i for i, (a, s) in enumerate(zip(agents, shots)) if a == ag and s == shot]
            avgs.append(np.mean(f1[idx]) if idx else 0)
        offset = (si - 0.5) * w
        bars = ax.bar(x + offset, avgs, width=w, color=color, alpha=0.85,
                      label=f"{shot}-shot", zorder=3)
        for bar, v in zip(bars, avgs):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.2,
                    f"{v:.1f}", ha="center", va="bottom", fontsize=8.5, color=TEXT_MID)

    ax.set_xticks(x)
    ax.set_xticklabels(["Dual Agent", "Single Agent", "No Agent"], color=TEXT_MID)
    ax.set_ylim(55, 78)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.set_title("F1 Score by Agent Architecture", color=TEXT, fontsize=13, pad=14, fontweight="bold")
    ax.set_ylabel("F1 Score (%)", color=TEXT_DIM)

    legend = ax.legend(frameon=True, fontsize=9, labelcolor=TEXT_MID)
    legend.get_frame().set_facecolor(SURFACE2)
    legend.get_frame().set_edgecolor(GRID)

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# Chart 6 — Stacked confusion matrix (DA configs only)
# ═══════════════════════════════════════════════════════════════════════════════
def chart6_confusion_stacked():
    da_idx  = [i for i, a in enumerate(agents) if a == "DA"]
    da_lbls = [f"{prompts[i]}\n{shots[i]}" for i in da_idx]

    fig, ax = plt.subplots(figsize=(14, 5))
    apply_dark_style(fig, ax)

    x = np.arange(len(da_idx))
    w = 0.6

    tp_v = tp[da_idx]
    fp_v = fp[da_idx]
    tn_v = tn[da_idx]
    fn_v = fn[da_idx]

    ax.bar(x, tp_v, width=w, color=C_NA,    alpha=0.85, label="TP", zorder=3)
    ax.bar(x, fp_v, width=w, bottom=tp_v,   color=C_REC,  alpha=0.75, label="FP", zorder=3)
    ax.bar(x, tn_v, width=w, bottom=tp_v+fp_v, color=C_PREC, alpha=0.75, label="TN", zorder=3)
    ax.bar(x, fn_v, width=w, bottom=tp_v+fp_v+tn_v, color=C_F1, alpha=0.85, label="FN", zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(da_lbls, fontsize=7.5, rotation=40, ha="right", color=TEXT_MID)
    ax.set_title("Confusion Counts — Dual Agent Configurations (Stacked)",
                 color=TEXT, fontsize=13, pad=14, fontweight="bold")
    ax.set_ylabel("Count", color=TEXT_DIM)

    legend = ax.legend(frameon=True, fontsize=9, labelcolor=TEXT_MID, ncol=4,
                       loc="upper right")
    legend.get_frame().set_facecolor(SURFACE2)
    legend.get_frame().set_edgecolor(GRID)

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# Save individual PDFs + combined PDF
# ═══════════════════════════════════════════════════════════════════════════════
charts = [
    ("chart1_all_metrics",    chart1_all_metrics,    "All Metrics — All Configurations"),
    ("chart2_precision_recall_scatter", chart2_scatter, "Precision vs Recall Scatter"),
    ("chart3_fewshot_vs_zeroshot",      chart3_shot_compare, "Few-shot vs Zero-shot"),
    ("chart4_agent_accuracy", chart4_agent_accuracy, "Agent Architecture — Accuracy"),
    ("chart5_agent_f1",       chart5_agent_f1,       "Agent Architecture — F1"),
    ("chart6_confusion_stacked",        chart6_confusion_stacked, "Confusion Counts (DA)"),
]

pdf_paths = []
for name, fn_chart, title in charts:
    path = os.path.join(OUT_DIR, f"{name}.pdf")
    fig  = fn_chart()
    fig.savefig(path, format="pdf", bbox_inches="tight",
                facecolor=BG, dpi=150)
    plt.close(fig)
    pdf_paths.append(path)
    print(f"  Saved {path}")

# ── Combined PDF (all 6 pages) ────────────────────────────────────────────────
combined_path = os.path.join(OUT_DIR, "log_analysis_all_charts.pdf")
with PdfPages(combined_path) as pdf:
    for name, fn_chart, title in charts:
        fig = fn_chart()
        pdf.savefig(fig, bbox_inches="tight", facecolor=BG, dpi=150)
        plt.close(fig)
        print(f"  Added '{title}' to combined PDF")

    # metadata
    d = pdf.infodict()
    d["Title"]   = "iExplain — Log Anomaly Detection Results"
    d["Subject"] = "HDFS 100-session experiment results"

print(f"\nCombined PDF → {combined_path}")
print("Done.")

  Saved /Users/merveastekin/Desktop/iExplain-logAnalysis/plots/chart1_all_metrics.pdf
  Saved /Users/merveastekin/Desktop/iExplain-logAnalysis/plots/chart2_precision_recall_scatter.pdf
  Saved /Users/merveastekin/Desktop/iExplain-logAnalysis/plots/chart3_fewshot_vs_zeroshot.pdf
  Saved /Users/merveastekin/Desktop/iExplain-logAnalysis/plots/chart4_agent_accuracy.pdf
  Saved /Users/merveastekin/Desktop/iExplain-logAnalysis/plots/chart5_agent_f1.pdf
  Saved /Users/merveastekin/Desktop/iExplain-logAnalysis/plots/chart6_confusion_stacked.pdf
  Added 'All Metrics — All Configurations' to combined PDF
  Added 'Precision vs Recall Scatter' to combined PDF
  Added 'Few-shot vs Zero-shot' to combined PDF
  Added 'Agent Architecture — Accuracy' to combined PDF
  Added 'Agent Architecture — F1' to combined PDF
  Added 'Confusion Counts (DA)' to combined PDF

Combined PDF → /Users/merveastekin/Desktop/iExplain-logAnalysis/plots/log_analysis_all_charts.pdf
Done.


In [ ]:
"""
iExplain — Log Analysis Experiment Results
Generates 6 chart PDFs.
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import os

# ── Output directory ──────────────────────────────────────────────────────────
OUT_DIR = "/Users/merveastekin/Desktop/iExplain-logAnalysis/plots"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Colour palette (matches dashboard) ───────────────────────────────────────
C_ACCENT  = "#5effd8"   # teal
C_PREC    = "#818cf8"   # purple
C_REC     = "#ff5e8a"   # pink/red
C_F1      = "#ffe05e"   # amber
C_SA      = "#fbbf24"   # yellow (Single Agent)
C_NA      = "#34d399"   # green  (No Agent)
C_FEW     = C_ACCENT
C_ZERO    = C_REC
BG        = "#0a0c10"
SURFACE   = "#111318"
SURFACE2  = "#181b22"
GRID      = "#252830"
TEXT      = "#e8ecf0"
TEXT_DIM  = "#6b7280"
TEXT_MID  = "#9ca3af"

def apply_dark_style(fig, axes=None):
    fig.patch.set_facecolor(BG)
    if axes is None:
        axes = fig.get_axes()
    for ax in (axes if hasattr(axes, '__iter__') else [axes]):
        ax.set_facecolor(SURFACE)
        ax.tick_params(colors=TEXT_MID, labelsize=9)
        ax.xaxis.label.set_color(TEXT_DIM)
        ax.yaxis.label.set_color(TEXT_DIM)
        ax.title.set_color(TEXT)
        for spine in ax.spines.values():
            spine.set_edgecolor(GRID)
        ax.grid(color=GRID, linewidth=0.6, linestyle="--", alpha=0.8)
        ax.set_axisbelow(True)

# ── Data ──────────────────────────────────────────────────────────────────────
data = [
    # prompt,           agent, shot,    acc,   prec,   rec,    f1,    tp, fp, tn, fn
    ("SIMPLIFIED",      "DA",  "Few",   57,  51.72,  97.83, 67.67,  45, 42, 12,  1),
    ("SIMPLIFIED",      "DA",  "Zero",  53,  49.33,  80.43, 61.16,  37, 38, 16,  9),
    ("DETAILED",        "DA",  "Few",   57,  51.90,  89.13, 65.60,  41, 38, 16,  5),
    ("DETAILED",        "DA",  "Zero",  54,  50.00,  78.26, 61.02,  36, 36, 18, 10),
    ("MINIMAL_GEN",     "DA",  "Few",   49,  46.75,  78.26, 58.54,  36, 41, 13, 10),
    ("MINIMAL_GEN",     "DA",  "Zero",  62,  56.67,  73.91, 64.15,  34, 26, 28, 12),
    ("MINIMAL_GEN†",    "DA",  "Zero†", 60,  55.17,  69.57, 61.54,  32, 26, 28, 14),
    ("HDFS",            "DA",  "Few",   56,  51.11, 100.00, 67.65,  46, 44, 10,  0),
    ("HDFS",            "DA",  "Zero",  57,  51.69, 100.00, 68.15,  46, 43, 11,  0),
    ("HDFS†",           "DA",  "Zero†", 57,  51.69, 100.00, 68.15,  46, 43, 11,  0),
    ("CoT",             "DA",  "Few",   57,  51.69, 100.00, 68.15,  46, 43, 11,  0),
    ("CoT",             "DA",  "Zero",  52,  48.84,  91.30, 63.64,  42, 44, 10,  4),
    ("OLD",             "DA",  "Few",   56,  51.52,  73.91, 60.71,  34, 32, 22, 12),
    ("OLD",             "DA",  "Zero",  65,  59.65,  73.91, 66.02,  34, 23, 31, 12),
    ("OLD†",            "DA",  "Zero†", 62,  57.14,  69.57, 62.75,  32, 24, 30, 14),
    # Single Agent
    ("—",               "SA",  "Few",   65,  61.22,  65.22, 63.16,  30, 19, 35, 16),
    ("—",               "SA",  "Zero",  75,  83.87,  56.52, 67.53,  26,  5, 49, 20),
    # No Agent
    ("—",               "NA",  "Few",   73,  69.39,  73.91, 71.58,  34, 15, 39, 12),
    ("—",               "NA",  "Zero",  74,  81.25,  56.52, 66.67,  26,  6, 48, 20),
]

# Unpack into arrays
prompts = [d[0] for d in data]
agents  = [d[1] for d in data]
shots   = [d[2] for d in data]
acc     = np.array([d[3]  for d in data], dtype=float)
prec    = np.array([d[4]  for d in data], dtype=float)
rec     = np.array([d[5]  for d in data], dtype=float)
f1      = np.array([d[6]  for d in data], dtype=float)
tp      = np.array([d[7]  for d in data], dtype=float)
fp      = np.array([d[8]  for d in data], dtype=float)
tn      = np.array([d[9]  for d in data], dtype=float)
fn      = np.array([d[10] for d in data], dtype=float)

x_labels = [f"{p}\n{a}·{s}" for p, a, s in zip(prompts, agents, shots)]


# ═══════════════════════════════════════════════════════════════════════════════
# Chart 1 — All metrics bar chart
# ═══════════════════════════════════════════════════════════════════════════════
def chart1_all_metrics():
    fig, ax = plt.subplots(figsize=(18, 6))
    apply_dark_style(fig, ax)

    n   = len(data)
    x   = np.arange(n)
    w   = 0.2

    ax.bar(x - 1.5*w, acc,  width=w, color=C_ACCENT, alpha=0.85, label="Accuracy",  zorder=3)
    ax.bar(x - 0.5*w, prec, width=w, color=C_PREC,   alpha=0.85, label="Precision", zorder=3)
    ax.bar(x + 0.5*w, rec,  width=w, color=C_REC,    alpha=0.85, label="Recall",    zorder=3)
    ax.bar(x + 1.5*w, f1,   width=w, color=C_F1,     alpha=0.85, label="F1",        zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, fontsize=7, rotation=45, ha="right", color=TEXT_MID)
    ax.set_ylim(40, 108)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.set_title("Accuracy · Precision · Recall · F1 — All Configurations",
                 color=TEXT, fontsize=13, pad=14, fontweight="bold")
    ax.set_ylabel("Score (%)", color=TEXT_DIM)

    legend = ax.legend(frameon=True, fontsize=9, labelcolor=TEXT_MID)
    legend.get_frame().set_facecolor(SURFACE2)
    legend.get_frame().set_edgecolor(GRID)

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# Chart 2 — Precision vs Recall scatter
# ═══════════════════════════════════════════════════════════════════════════════
def chart2_scatter():
    fig, ax = plt.subplots(figsize=(8, 6))
    apply_dark_style(fig, ax)

    agent_cfg = {
        "DA": (C_PREC,  "o", "Dual Agent"),
        "SA": (C_SA,    "s", "Single Agent"),
        "NA": (C_NA,    "^", "No Agent"),
    }

    for ag, (color, marker, label) in agent_cfg.items():
        idx = [i for i, a in enumerate(agents) if a == ag]
        ax.scatter(prec[idx], rec[idx],
                   c=color, marker=marker, s=90, alpha=0.9,
                   label=label, zorder=4, edgecolors="white", linewidths=0.4)

    # annotate a few notable points
    notable = [
        (15, "SA·Zero"),   # best accuracy
        (17, "NA·Few"),    # best F1
        (0,  "SIMPL·Few"), # high recall
    ]
    for row_idx, lbl in notable:
        ax.annotate(lbl,
                    xy=(prec[row_idx], rec[row_idx]),
                    xytext=(6, 4), textcoords="offset points",
                    fontsize=7.5, color=TEXT_MID)

    ax.set_xlabel("Precision (%)", fontsize=10)
    ax.set_ylabel("Recall (%)",    fontsize=10)
    ax.set_title("Precision vs Recall — by Agent Type",
                 color=TEXT, fontsize=13, pad=14, fontweight="bold")
    ax.set_xlim(43, 88)
    ax.set_ylim(50, 107)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))

    legend = ax.legend(frameon=True, fontsize=9, labelcolor=TEXT_MID)
    legend.get_frame().set_facecolor(SURFACE2)
    legend.get_frame().set_edgecolor(GRID)

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# Chart 3 — Few-shot vs Zero-shot average metrics
# ═══════════════════════════════════════════════════════════════════════════════
def chart3_shot_compare():
    fig, ax = plt.subplots(figsize=(7, 5))
    apply_dark_style(fig, ax)

    few_idx  = [i for i, s in enumerate(shots) if s == "Few"]
    zero_idx = [i for i, s in enumerate(shots) if s == "Zero"]

    metrics = ["Accuracy", "Precision", "Recall", "F1"]
    few_avgs  = [np.mean(arr[few_idx])  for arr in (acc, prec, rec, f1)]
    zero_avgs = [np.mean(arr[zero_idx]) for arr in (acc, prec, rec, f1)]

    x = np.arange(len(metrics))
    w = 0.35

    ax.bar(x - w/2, few_avgs,  width=w, color=C_FEW,  alpha=0.85, label="Few-shot",  zorder=3)
    ax.bar(x + w/2, zero_avgs, width=w, color=C_ZERO, alpha=0.85, label="Zero-shot", zorder=3)

    # value labels
    for xi, (fv, zv) in enumerate(zip(few_avgs, zero_avgs)):
        ax.text(xi - w/2, fv + 0.5, f"{fv:.1f}", ha="center", va="bottom",
                fontsize=8, color=TEXT_MID)
        ax.text(xi + w/2, zv + 0.5, f"{zv:.1f}", ha="center", va="bottom",
                fontsize=8, color=TEXT_MID)

    ax.set_xticks(x)
    ax.set_xticklabels(metrics, color=TEXT_MID)
    ax.set_ylim(50, 95)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.set_title("Few-shot vs Zero-shot — Average Metrics",
                 color=TEXT, fontsize=13, pad=14, fontweight="bold")
    ax.set_ylabel("Score (%)", color=TEXT_DIM)

    legend = ax.legend(frameon=True, fontsize=9, labelcolor=TEXT_MID)
    legend.get_frame().set_facecolor(SURFACE2)
    legend.get_frame().set_edgecolor(GRID)

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# Chart 4 — Agent type accuracy comparison
# ═══════════════════════════════════════════════════════════════════════════════
def chart4_agent_accuracy():
    fig, ax = plt.subplots(figsize=(7, 5))
    apply_dark_style(fig, ax)

    agent_types = ["DA", "SA", "NA"]
    shot_types  = ["Few", "Zero"]
    x = np.arange(len(agent_types))
    w = 0.35

    for si, (shot, color) in enumerate(zip(shot_types, [C_FEW, C_ZERO])):
        avgs = []
        for ag in agent_types:
            idx = [i for i, (a, s) in enumerate(zip(agents, shots)) if a == ag and s == shot]
            avgs.append(np.mean(acc[idx]) if idx else 0)
        offset = (si - 0.5) * w
        bars = ax.bar(x + offset, avgs, width=w, color=color, alpha=0.85,
                      label=f"{shot}-shot", zorder=3)
        for bar, v in zip(bars, avgs):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.4,
                    f"{v:.1f}", ha="center", va="bottom", fontsize=8.5, color=TEXT_MID)

    ax.set_xticks(x)
    ax.set_xticklabels(["Dual Agent", "Single Agent", "No Agent"], color=TEXT_MID)
    ax.set_ylim(45, 85)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.set_title("Accuracy by Agent Architecture", color=TEXT, fontsize=13, pad=14, fontweight="bold")
    ax.set_ylabel("Accuracy (%)", color=TEXT_DIM)

    legend = ax.legend(frameon=True, fontsize=9, labelcolor=TEXT_MID)
    legend.get_frame().set_facecolor(SURFACE2)
    legend.get_frame().set_edgecolor(GRID)

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# Chart 5 — Agent type F1 comparison
# ═══════════════════════════════════════════════════════════════════════════════
def chart5_agent_f1():
    fig, ax = plt.subplots(figsize=(7, 5))
    apply_dark_style(fig, ax)

    agent_types = ["DA", "SA", "NA"]
    shot_types  = ["Few", "Zero"]
    x = np.arange(len(agent_types))
    w = 0.35

    for si, (shot, color) in enumerate(zip(shot_types, [C_FEW, C_ZERO])):
        avgs = []
        for ag in agent_types:
            idx = [i for i, (a, s) in enumerate(zip(agents, shots)) if a == ag and s == shot]
            avgs.append(np.mean(f1[idx]) if idx else 0)
        offset = (si - 0.5) * w
        bars = ax.bar(x + offset, avgs, width=w, color=color, alpha=0.85,
                      label=f"{shot}-shot", zorder=3)
        for bar, v in zip(bars, avgs):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.2,
                    f"{v:.1f}", ha="center", va="bottom", fontsize=8.5, color=TEXT_MID)

    ax.set_xticks(x)
    ax.set_xticklabels(["Dual Agent", "Single Agent", "No Agent"], color=TEXT_MID)
    ax.set_ylim(55, 78)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.set_title("F1 Score by Agent Architecture", color=TEXT, fontsize=13, pad=14, fontweight="bold")
    ax.set_ylabel("F1 Score (%)", color=TEXT_DIM)

    legend = ax.legend(frameon=True, fontsize=9, labelcolor=TEXT_MID)
    legend.get_frame().set_facecolor(SURFACE2)
    legend.get_frame().set_edgecolor(GRID)

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# Chart 6 — Stacked confusion matrix (DA configs only)
# ═══════════════════════════════════════════════════════════════════════════════
def chart6_confusion_stacked():
    da_idx  = [i for i, a in enumerate(agents) if a == "DA"]
    da_lbls = [f"{prompts[i]}\n{shots[i]}" for i in da_idx]

    fig, ax = plt.subplots(figsize=(14, 5))
    apply_dark_style(fig, ax)

    x = np.arange(len(da_idx))
    w = 0.6

    tp_v = tp[da_idx]
    fp_v = fp[da_idx]
    tn_v = tn[da_idx]
    fn_v = fn[da_idx]

    ax.bar(x, tp_v, width=w, color=C_NA,    alpha=0.85, label="TP", zorder=3)
    ax.bar(x, fp_v, width=w, bottom=tp_v,   color=C_REC,  alpha=0.75, label="FP", zorder=3)
    ax.bar(x, tn_v, width=w, bottom=tp_v+fp_v, color=C_PREC, alpha=0.75, label="TN", zorder=3)
    ax.bar(x, fn_v, width=w, bottom=tp_v+fp_v+tn_v, color=C_F1, alpha=0.85, label="FN", zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(da_lbls, fontsize=7.5, rotation=40, ha="right", color=TEXT_MID)
    ax.set_title("Confusion Counts — Dual Agent Configurations (Stacked)",
                 color=TEXT, fontsize=13, pad=14, fontweight="bold")
    ax.set_ylabel("Count", color=TEXT_DIM)

    legend = ax.legend(frameon=True, fontsize=9, labelcolor=TEXT_MID, ncol=4,
                       loc="upper right")
    legend.get_frame().set_facecolor(SURFACE2)
    legend.get_frame().set_edgecolor(GRID)

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# DA Deep-Dive — helper: DA-only filtered arrays
# ═══════════════════════════════════════════════════════════════════════════════
BEST_CONFIG   = "OLD"           # recommended prompt type
BEST_SHOT     = "Zero"          # recommended shot type
HIGHLIGHT_CLR = "#ffe05e"       # amber highlight for best config

da_idx   = [i for i, a in enumerate(agents) if a == "DA"]
da_proms = [prompts[i] for i in da_idx]
da_shots = [shots[i]   for i in da_idx]
da_lbls  = [f"{prompts[i]}\n{shots[i]}" for i in da_idx]

da_acc  = acc[da_idx];  da_prec = prec[da_idx]
da_rec  = rec[da_idx];  da_f1   = f1[da_idx]
da_tp   = tp[da_idx];   da_fp   = fp[da_idx]
da_tn   = tn[da_idx];   da_fn   = fn[da_idx]

def bar_colors_da(base_color, highlight_indices):
    """Return per-bar color list, highlighting selected bars."""
    colors = [base_color] * len(da_idx)
    for hi in highlight_indices:
        colors[hi] = HIGHLIGHT_CLR
    return colors

# Index of OLD-Zero within DA subset
best_idx_da = next(i for i, (p, s) in enumerate(zip(da_proms, da_shots))
                   if p == BEST_CONFIG and s == BEST_SHOT)


# ═══════════════════════════════════════════════════════════════════════════════
# DA Chart 1 — All four metrics side-by-side, DA only, best config highlighted
# ═══════════════════════════════════════════════════════════════════════════════
def da_chart1_metrics():
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    apply_dark_style(fig, axes.flat)
    fig.suptitle("Dual Agent — Metric Breakdown per Configuration",
                 color=TEXT, fontsize=14, fontweight="bold", y=1.01)

    metric_data = [
        (da_acc,  "Accuracy (%)",  C_ACCENT, (40, 70)),
        (da_prec, "Precision (%)", C_PREC,   (40, 65)),
        (da_rec,  "Recall (%)",    C_REC,    (60, 108)),
        (da_f1,   "F1 Score (%)",  C_F1,     (50, 75)),
    ]

    x = np.arange(len(da_idx))
    for ax, (vals, ylabel, color, ylim) in zip(axes.flat, metric_data):
        bar_clrs = [HIGHLIGHT_CLR if i == best_idx_da else color for i in range(len(da_idx))]
        bars = ax.bar(x, vals, color=bar_clrs, alpha=0.88, width=0.6, zorder=3)

        # value labels on bars
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.4,
                    f"{v:.1f}", ha="center", va="bottom", fontsize=7.5, color=TEXT_MID)

        ax.set_xticks(x)
        ax.set_xticklabels(da_lbls, fontsize=7, rotation=40, ha="right", color=TEXT_MID)
        ax.set_ylim(*ylim)
        ax.set_ylabel(ylabel, color=TEXT_DIM, fontsize=9)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))

        # best config annotation
        bx = bars[best_idx_da]
        ax.annotate("★ Best",
                    xy=(bx.get_x() + bx.get_width()/2, vals[best_idx_da]),
                    xytext=(0, 14), textcoords="offset points",
                    ha="center", fontsize=8, color=HIGHLIGHT_CLR,
                    arrowprops=dict(arrowstyle="-", color=HIGHLIGHT_CLR, lw=0.8))

    # legend patch for highlight
    patch = mpatches.Patch(color=HIGHLIGHT_CLR, label=f"★ Recommended: {BEST_CONFIG} · {BEST_SHOT}-shot")
    fig.legend(handles=[patch], loc="lower center", fontsize=9,
               frameon=True, labelcolor=TEXT_MID,
               facecolor=SURFACE2, edgecolor=GRID, ncol=1, bbox_to_anchor=(0.5, -0.02))

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# DA Chart 2 — Few vs Zero within DA only (grouped bar, all 4 metrics)
# ═══════════════════════════════════════════════════════════════════════════════
def da_chart2_few_vs_zero():
    fig, ax = plt.subplots(figsize=(8, 5))
    apply_dark_style(fig, ax)

    few_da  = [i for i, s in enumerate(da_shots) if s == "Few"]
    zero_da = [i for i, s in enumerate(da_shots) if s in ("Zero", "Zero†")]

    metrics   = ["Accuracy", "Precision", "Recall", "F1"]
    few_avgs  = [np.mean(arr[few_da])  for arr in (da_acc, da_prec, da_rec, da_f1)]
    zero_avgs = [np.mean(arr[zero_da]) for arr in (da_acc, da_prec, da_rec, da_f1)]

    x = np.arange(len(metrics))
    w = 0.35
    b1 = ax.bar(x - w/2, few_avgs,  width=w, color=C_FEW,  alpha=0.85, label="Few-shot",  zorder=3)
    b2 = ax.bar(x + w/2, zero_avgs, width=w, color=C_ZERO, alpha=0.85, label="Zero-shot", zorder=3)

    for bar, v in list(zip(b1, few_avgs)) + list(zip(b2, zero_avgs)):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.3,
                f"{v:.1f}", ha="center", va="bottom", fontsize=8.5, color=TEXT_MID)

    ax.set_xticks(x)
    ax.set_xticklabels(metrics, color=TEXT_MID)
    ax.set_ylim(50, 100)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.set_title("Dual Agent — Few-shot vs Zero-shot (Average Metrics)",
                 color=TEXT, fontsize=13, pad=14, fontweight="bold")
    ax.set_ylabel("Score (%)", color=TEXT_DIM)

    legend = ax.legend(frameon=True, fontsize=9, labelcolor=TEXT_MID)
    legend.get_frame().set_facecolor(SURFACE2)
    legend.get_frame().set_edgecolor(GRID)

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# DA Chart 3 — Precision / Recall trade-off scatter (DA only), annotated
# ═══════════════════════════════════════════════════════════════════════════════
def da_chart3_pr_scatter():
    fig, ax = plt.subplots(figsize=(9, 6))
    apply_dark_style(fig, ax)

    shot_cfg = {
        "Few":   (C_FEW,  "o"),
        "Zero":  (C_ZERO, "s"),
        "Zero†": (C_PREC, "^"),
    }

    for shot_key, (color, marker) in shot_cfg.items():
        idx = [i for i, s in enumerate(da_shots) if s == shot_key]
        if not idx:
            continue
        ax.scatter(da_prec[idx], da_rec[idx],
                   c=color, marker=marker, s=100, alpha=0.9,
                   label=f"{shot_key}-shot", zorder=4,
                   edgecolors="white", linewidths=0.5)

    # annotate every point
    for i, (p, r, lbl) in enumerate(zip(da_prec, da_rec, da_lbls)):
        clean_lbl = lbl.replace("\n", " ")
        is_best   = (da_proms[i] == BEST_CONFIG and da_shots[i] == BEST_SHOT)
        ax.annotate(("★ " if is_best else "") + clean_lbl,
                    xy=(p, r), xytext=(5, 4 if i % 2 == 0 else -12),
                    textcoords="offset points",
                    fontsize=7, color=HIGHLIGHT_CLR if is_best else TEXT_MID,
                    fontweight="bold" if is_best else "normal")

    # ideal region box
    ax.axvspan(58, 90, alpha=0.04, color=C_ACCENT, zorder=0)
    ax.axhspan(65, 105, alpha=0.04, color=C_ACCENT, zorder=0)
    ax.text(58.5, 104, "Higher precision zone", fontsize=7.5, color=C_ACCENT, alpha=0.6)

    ax.set_xlabel("Precision (%)", fontsize=10)
    ax.set_ylabel("Recall (%)",    fontsize=10)
    ax.set_xlim(44, 64)
    ax.set_ylim(62, 108)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.set_title("Dual Agent — Precision vs Recall Trade-off",
                 color=TEXT, fontsize=13, pad=14, fontweight="bold")

    legend = ax.legend(frameon=True, fontsize=9, labelcolor=TEXT_MID)
    legend.get_frame().set_facecolor(SURFACE2)
    legend.get_frame().set_edgecolor(GRID)

    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# DA Chart 4 — Individual confusion matrices (2×8 grid) with best highlighted
# ═══════════════════════════════════════════════════════════════════════════════
def da_chart4_confusion_grid():
    n     = len(da_idx)
    ncols = 5
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.4))
    fig.patch.set_facecolor(BG)
    fig.suptitle("Dual Agent — Individual Confusion Matrices  (★ = Recommended)",
                 color=TEXT, fontsize=13, fontweight="bold", y=1.01)

    for i, ax in enumerate(axes.flat):
        ax.set_facecolor(SURFACE)
        for spine in ax.spines.values():
            spine.set_edgecolor(GRID)

        if i >= n:
            ax.axis("off")
            continue

        is_best = (i == best_idx_da)
        matrix  = np.array([[da_tp[i], da_fn[i]],
                             [da_fp[i], da_tn[i]]])
        labels  = [["TP", "FN"], ["FP", "TN"]]
        colors  = [[C_NA, C_F1], [C_REC, C_PREC]]

        for ri in range(2):
            for ci in range(2):
                val = int(matrix[ri, ci])
                rect = mpatches.FancyBboxPatch(
                    (ci * 0.5 + 0.02, (1 - ri) * 0.5 + 0.02),
                    0.46, 0.46,
                    boxstyle="round,pad=0.02",
                    facecolor=colors[ri][ci],
                    alpha=0.25 if not is_best else 0.45,
                    transform=ax.transAxes, zorder=2
                )
                ax.add_patch(rect)
                ax.text(ci * 0.5 + 0.25, (1 - ri) * 0.5 + 0.28,
                        str(val),
                        ha="center", va="center",
                        fontsize=16, fontweight="bold",
                        color=colors[ri][ci],
                        transform=ax.transAxes, zorder=3)
                ax.text(ci * 0.5 + 0.25, (1 - ri) * 0.5 + 0.12,
                        labels[ri][ci],
                        ha="center", va="center",
                        fontsize=8, color=TEXT_DIM,
                        transform=ax.transAxes, zorder=3)

        border_color = HIGHLIGHT_CLR if is_best else GRID
        for spine in ax.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(2 if is_best else 0.8)

        title_str = ("★ " if is_best else "") + da_lbls[i].replace("\n", " · ")
        ax.set_title(title_str,
                     fontsize=8, color=HIGHLIGHT_CLR if is_best else TEXT_MID,
                     fontweight="bold" if is_best else "normal", pad=5)

        # small metric row below
        ax.text(0.5, -0.08,
                f"Acc {int(da_acc[i])}%  F1 {da_f1[i]:.1f}%",
                ha="center", va="top", fontsize=7.5,
                color=TEXT_DIM, transform=ax.transAxes)

        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_xticks([]); ax.set_yticks([])

        # row/col header labels (only first row/col)
        if i < ncols:
            ax.text(0.25, 1.02, "Pred +", ha="center", fontsize=7,
                    color=TEXT_DIM, transform=ax.transAxes)
            ax.text(0.75, 1.02, "Pred −", ha="center", fontsize=7,
                    color=TEXT_DIM, transform=ax.transAxes)

    fig.tight_layout(h_pad=2.5, w_pad=1.5)
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# Save individual PDFs + combined PDF (original 6 + 4 DA deep-dive pages)
# ═══════════════════════════════════════════════════════════════════════════════
charts = [
    ("chart1_all_metrics",              chart1_all_metrics,    "All Metrics — All Configurations"),
    ("chart2_precision_recall_scatter", chart2_scatter,        "Precision vs Recall Scatter"),
    ("chart3_fewshot_vs_zeroshot",      chart3_shot_compare,   "Few-shot vs Zero-shot"),
    ("chart4_agent_accuracy",           chart4_agent_accuracy, "Agent Architecture — Accuracy"),
    ("chart5_agent_f1",                 chart5_agent_f1,       "Agent Architecture — F1"),
    ("chart6_confusion_stacked",        chart6_confusion_stacked, "Confusion Counts (DA)"),
    # ── DA deep-dive ──────────────────────────────────────────────────────────
    ("da_chart1_metrics",               da_chart1_metrics,     "DA — Metric Breakdown (Highlighted)"),
    ("da_chart2_few_vs_zero",           da_chart2_few_vs_zero, "DA — Few-shot vs Zero-shot"),
    ("da_chart3_pr_scatter",            da_chart3_pr_scatter,  "DA — Precision vs Recall Trade-off"),
    ("da_chart4_confusion_grid",        da_chart4_confusion_grid, "DA — Individual Confusion Matrices"),
]

pdf_paths = []
for name, fn_chart, title in charts:
    path = os.path.join(OUT_DIR, f"{name}.pdf")
    fig  = fn_chart()
    fig.savefig(path, format="pdf", bbox_inches="tight", facecolor=BG, dpi=150)
    plt.close(fig)
    pdf_paths.append(path)
    print(f"  Saved {path}")

# ── Combined PDF (10 pages) ───────────────────────────────────────────────────
combined_path = os.path.join(OUT_DIR, "log_analysis_all_charts.pdf")
with PdfPages(combined_path) as pdf:
    for name, fn_chart, title in charts:
        fig = fn_chart()
        pdf.savefig(fig, bbox_inches="tight", facecolor=BG, dpi=150)
        plt.close(fig)
        print(f"  Added '{title}' to combined PDF")

    d = pdf.infodict()
    d["Title"]   = "iExplain — Log Anomaly Detection Results"
    d["Subject"] = "HDFS 100-session experiment results — incl. DA deep-dive"

print(f"\nCombined PDF ({len(charts)} pages) → {combined_path}")
print("Done.")